# Binary Ensemble Training (Merged Dataset)

This notebook trains a **binary ONNX ensemble** (Cancer vs Non-cancer) on the **merged dataset**, using the same architecture and patterns as the 5-class ensemble in `pytorch_to_onnx_ensemble.ipynb`.

## Goal
- Train **3 PyTorch models** (EnsembleModel1, 2, 3) with **num_classes=2** on merged binary data.
- Data: binary train+val (TB excluded) + 5-class train+val mapped to binary (T5 excluded; Unknown excluded).
- Export to ONNX for use in the app alongside the 5-class ensemble (see `DUAL_ENSEMBLE_APP_PLAN.md`).

## Pipeline
1. Configuration and setup
2. Model architecture (ConservativeMobileNetV3 + EnsembleModel1/2/3, num_classes=2)
3. Load merged dataset (frozen_test_sets)
4. Train/val split, Dataset, DataLoaders
5. Train 3 ensemble models (CrossEntropyLoss, Adam, StepLR, early stopping)
6. Save PyTorch checkpoints → Convert to ONNX
7. Evaluate on frozen T5/TB/TM

**App assets:** The Flutter app loads binary ONNX from `assets/binary_ensemble/`. Copy from the canonical run; see **`results/binary_ensemble/ASSETS_SOURCE.md`** (current: **run_20260130_135331/onnx**).

## 1. Configuration

In [1]:
# Configuration
import os
from pathlib import Path
from datetime import datetime
import json

# Paths - update to match your setup
DATA_DIR_5CLASS = os.path.expanduser("~/dev/oral_cavity_5class_dataset")
DATA_DIR_BINARY = os.path.expanduser("~/dev/oral_cavity_binary_dataset")
NOTEBOOK_DIR = Path(".").resolve()
FROZEN_DIR = NOTEBOOK_DIR / "frozen_test_sets"

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
run_name = os.environ.get("BINARY_EXPERIMENT_NAME") or f"run_{timestamp}"

# Training hyperparameters (aligned with production ensemble)
DEFAULT_CONFIG = {
    "image_size": (224, 224),
    "batch_size": 32,
    "learning_rate": 1e-4,
    "num_epochs": 30,
    "patience": 8,
    "val_split": 0.15,
    "num_workers": 0,
    "seed": 42,
    "augmentation": "basic",
    "scheduler": "step",
    "binary_threshold": 0.5,
    "cancer_weight_multiplier": 1.0,
}

config_path = os.environ.get("BINARY_EXPERIMENT_CONFIG")
if config_path and Path(config_path).exists():
    with open(config_path, "r") as f:
        overrides = json.load(f)
    DEFAULT_CONFIG.update(overrides)

CONFIG = DEFAULT_CONFIG
run_name = CONFIG.get("run_name", run_name)

# Output: timestamped run folder (PyTorch + ONNX)
OUTPUT_DIR = NOTEBOOK_DIR / f"results/binary_ensemble/{run_name}"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / "models").mkdir(exist_ok=True)
(OUTPUT_DIR / "onnx").mkdir(exist_ok=True)

with open(OUTPUT_DIR / "config.json", "w") as f:
    json.dump({"data_dir_5class": DATA_DIR_5CLASS, "data_dir_binary": DATA_DIR_BINARY, **CONFIG}, f, indent=2)

print("✅ Configuration loaded")
print(f"📁 5-class data: {DATA_DIR_5CLASS}")
print(f"📁 Binary data: {DATA_DIR_BINARY}")
print(f"📁 Frozen sets: {FROZEN_DIR}")
print(f"📁 Output: {OUTPUT_DIR}")
print(f"   Epochs: {CONFIG['num_epochs']}, Patience: {CONFIG['patience']}, LR: {CONFIG['learning_rate']}")
print(f"   Augmentation: {CONFIG['augmentation']}, Scheduler: {CONFIG['scheduler']}, Threshold: {CONFIG['binary_threshold']}")


✅ Configuration loaded
📁 5-class data: /Users/alokdwivedi/dev/oral_cavity_5class_dataset
📁 Binary data: /Users/alokdwivedi/dev/oral_cavity_binary_dataset
📁 Frozen sets: /Users/alokdwivedi/dev/github/3amwave/oral-cavity-classifier/ml/training/notebooks/frozen_test_sets
📁 Output: /Users/alokdwivedi/dev/github/3amwave/oral-cavity-classifier/ml/training/notebooks/results/binary_ensemble/run_20260203_213434
   Epochs: 30, Patience: 8, LR: 0.0001
   Augmentation: basic, Scheduler: step, Threshold: 0.5


## 2. Setup and device

In [2]:
# Install (uncomment if needed)
# !pip install torch torchvision onnx onnxruntime pillow numpy scikit-learn tqdm


In [3]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import torchvision.transforms as transforms
from torchvision import models
from PIL import Image
import numpy as np
from sklearn.model_selection import train_test_split
from tqdm import tqdm
import json
import warnings
warnings.filterwarnings("ignore")

try:
    import onnx
    import onnxruntime as ort
    HAS_ONNX = True
except ImportError:
    HAS_ONNX = False

print("✅ Dependencies loaded")
print(f"🔥 PyTorch: {torch.__version__}")
if HAS_ONNX:
    print(f"🔥 ONNX: {onnx.__version__}, ONNX Runtime: {ort.__version__}")

✅ Dependencies loaded
🔥 PyTorch: 2.10.0
🔥 ONNX: 1.20.1, ONNX Runtime: 1.23.2


In [4]:
# Device
if torch.backends.mps.is_available():
    device = torch.device("mps")
    print(f"🍎 Using Apple Silicon GPU (MPS): {device}")
elif torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"🚀 Using CUDA: {device}")
else:
    device = torch.device("cpu")
    print(f"💻 Using CPU: {device}")

🍎 Using Apple Silicon GPU (MPS): mps


## 3. Model architecture (num_classes=2)

In [5]:
# Same architecture as pytorch_to_onnx_ensemble.ipynb but num_classes=2
NUM_CLASSES = 2  # 0 = non-cancer, 1 = cancer

class ConservativeMobileNetV3(nn.Module):
    """Conservative MobileNetV3Large - matches 5-class ensemble backbone."""

    def __init__(self, num_classes=2, dropout_rate=0.2):
        super().__init__()
        try:
            self.backbone = models.mobilenet_v3_large(weights=models.MobileNet_V3_Large_Weights.IMAGENET1K_V2)
        except AttributeError:
            self.backbone = models.mobilenet_v3_large(pretrained=True)
        self.backbone.classifier = nn.Identity()
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
            nn.Dropout(dropout_rate),
            nn.Linear(960, 256),
            nn.ReLU(inplace=True),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        x = self.backbone.features(x)
        x = self.classifier(x)
        return x


class EnsembleModel1(ConservativeMobileNetV3):
    def __init__(self, num_classes=2):
        super().__init__(num_classes=num_classes, dropout_rate=0.3)


class EnsembleModel2(ConservativeMobileNetV3):
    def __init__(self, num_classes=2):
        super().__init__(num_classes=num_classes, dropout_rate=0.1)


class EnsembleModel3(ConservativeMobileNetV3):
    def __init__(self, num_classes=2):
        super().__init__(num_classes=num_classes, dropout_rate=0.2)
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
            nn.Dropout(0.2),
            nn.Linear(960, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.1),
            nn.Linear(512, num_classes),
        )


ENSEMBLE_CLASSES = [EnsembleModel1, EnsembleModel2, EnsembleModel3]
print(f"✅ Model classes defined (num_classes={NUM_CLASSES})")

✅ Model classes defined (num_classes=2)


## 4. Load merged dataset (frozen_test_sets)

### Binary label spec (merged dataset)

**Intended mapping:**

- **Cancer (label 1):** OCA and OPMD  
  - From 5-class: `train/OCA`, `train/OPMD`, `val/OCA`, `val/OPMD`  
  - From binary dataset: `cancer/` folder  
- **Non-cancer (label 0):** Healthy and Benign  
  - From 5-class: `train/Healthy`, `train/Benign`, `val/Healthy`, `val/Benign`  
  - From binary dataset: `non-cancer/` folder  

**Unknown** is excluded (not used in merged binary training).

So for any **OPMD image** from the 5-class dataset, the binary ensemble is trained to predict **Cancer (1)**. If at inference time the binary model outputs low P(cancer) for a known OPMD image, possible causes: (1) **distribution shift** (e.g. phone photo vs clinical training images), (2) **model sensitivity** (binary head less discriminative on OPMD than 5-class), (3) **calibration**. The app’s dual-ensemble matrix corrects by elevating risk when the 5-class says OPMD/OCA even if the binary says non-cancer.

In [6]:
import sys
sys.path.insert(0, str(NOTEBOOK_DIR))
from frozen_test_sets import build_and_save_frozen_sets, get_binary_train_val_for_training

def collect_images(folder, exts=("*.jpg", "*.jpeg", "*.png", "*.JPG", "*.JPEG", "*.PNG")):
    folder = Path(folder)
    out = []
    if folder.exists():
        for ext in exts:
            out.extend([str(p) for p in folder.glob(ext)])
    return out

# Build frozen sets if needed
build_and_save_frozen_sets(
    data_dir_5class=DATA_DIR_5CLASS,
    data_dir_binary=DATA_DIR_BINARY,
    out_dir=FROZEN_DIR,
)

# Binary train+val (TB excluded)
binary_paths, binary_labels = get_binary_train_val_for_training(DATA_DIR_BINARY, FROZEN_DIR)
binary_cancer = [p for p, lb in zip(binary_paths, binary_labels) if lb == 1]
binary_non = [p for p, lb in zip(binary_paths, binary_labels) if lb == 0]

# 5-class train+val only (T5 excluded), mapped to binary; exclude Unknown
five_dir = Path(DATA_DIR_5CLASS)
five_cancer, five_non = [], []
for split in ["train", "val"]:
    for c in ["OCA", "OPMD"]:
        five_cancer.extend(collect_images(five_dir / split / c))
    for c in ["Benign", "Healthy"]:
        five_non.extend(collect_images(five_dir / split / c))

# Merge (dedup by path)
all_cancer = list(set(binary_cancer + five_cancer))
all_non = list(set(binary_non + five_non))
all_paths = all_cancer + all_non
all_labels = [1] * len(all_cancer) + [0] * len(all_non)

# Sanity: 5-class Cancer = OCA + OPMD (so OPMD is trained as Cancer)
five_cancer_opmd = [p for p in five_cancer if "OPMD" in p]
five_cancer_oca = [p for p in five_cancer if "OCA" in p]
print("Merged training pool:")
print(f"  Cancer: {len(all_cancer)} (binary: {len(binary_cancer)}, 5-class: {len(five_cancer)})")
print(f"    5-class Cancer breakdown: OPMD={len(five_cancer_opmd)}, OCA={len(five_cancer_oca)}")
print(f"  Non-cancer: {len(all_non)} (binary: {len(binary_non)}, 5-class: {len(five_non)})")
print(f"  Total: {len(all_paths)}")

Frozen test sets saved to /Users/alokdwivedi/dev/github/3amwave/oral-cavity-classifier/ml/training/notebooks/frozen_test_sets
  T5: 534 samples (5-class test)
  TB: 246 samples (binary-only test)
  TM: 730 samples (mixed, source-balanced for reporting)
  binary_train_val_for_training: 1388 paths (use these for merged training)
Merged training pool:
  Cancer: 2186 (binary: 751, 5-class: 1435)
  Non-cancer: 2040 (binary: 637, 5-class: 1403)
  Total: 4226


## 5. Train/val split, Dataset, DataLoaders

In [7]:
train_paths, val_paths, train_labels, val_labels = train_test_split(
    all_paths, all_labels,
    test_size=CONFIG["val_split"],
    random_state=CONFIG["seed"],
    stratify=all_labels,
)

print(f"Train: {len(train_paths)}, Val: {len(val_paths)}")

Train: 3592, Val: 634


In [8]:
# Transforms (ImageNet norm - same as pytorch_to_onnx_ensemble / evaluate_on_frozen_sets)
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

augmentation = CONFIG.get("augmentation", "basic")
train_transforms = [
    transforms.Resize(CONFIG["image_size"]),
]

if augmentation == "strong":
    train_transforms.extend([
        transforms.RandomResizedCrop(CONFIG["image_size"], scale=(0.7, 1.0)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(20),
        transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.05),
        transforms.RandomPerspective(distortion_scale=0.2, p=0.3),
        transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 2.0)),
    ])
else:
    train_transforms.extend([
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(15),
        transforms.ColorJitter(brightness=0.2, contrast=0.2),
    ])

train_transforms.extend([
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

train_transform = transforms.Compose(train_transforms)

val_transform = transforms.Compose([
    transforms.Resize(CONFIG["image_size"]),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])


class PathLabelDataset(Dataset):
    def __init__(self, paths, labels, transform=None):
        self.paths = paths
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        path = self.paths[idx]
        label = self.labels[idx]
        img = Image.open(path).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, label


train_ds = PathLabelDataset(train_paths, train_labels, transform=train_transform)
val_ds = PathLabelDataset(val_paths, val_labels, transform=val_transform)

train_loader = DataLoader(
    train_ds,
    batch_size=CONFIG["batch_size"],
    shuffle=True,
    num_workers=CONFIG["num_workers"],
    pin_memory=False,
)
val_loader = DataLoader(
    val_ds,
    batch_size=CONFIG["batch_size"],
    shuffle=False,
    num_workers=CONFIG["num_workers"],
)

print("✅ DataLoaders ready")


✅ DataLoaders ready


In [9]:
# Optional: class weights for imbalance
from sklearn.utils.class_weight import compute_class_weight
classes = np.unique(train_labels)
weights = compute_class_weight(
    "balanced",
    classes=classes,
    y=train_labels,
)
class_weights = torch.tensor(weights, dtype=torch.float32)

cancer_weight_multiplier = float(CONFIG.get("cancer_weight_multiplier", 1.0))
if cancer_weight_multiplier != 1.0:
    class_weights = class_weights.clone()
    class_weights[1] = class_weights[1] * cancer_weight_multiplier
    print(f"Applied cancer weight multiplier: {cancer_weight_multiplier}")

print(f"Class weights: {class_weights.tolist()}")


Class weights: [1.0357555150985718, 0.966630756855011]


## 6. Training loop (single ensemble model)

In [ ]:
def train_ensemble_model(model_class, model_name, train_loader, val_loader, device, config, class_weights=None):
    """Train one binary ensemble model; return best model state and val accuracy."""
    model = model_class(num_classes=NUM_CLASSES).to(device)
    criterion = nn.CrossEntropyLoss(weight=class_weights.to(device) if class_weights is not None else None)
    optimizer = optim.Adam(model.parameters(), lr=config["learning_rate"])
    scheduler_type = config.get("scheduler", "step")
    if scheduler_type == "cosine":
        scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=config["num_epochs"])
    elif scheduler_type == "none":
        scheduler = None
    else:
        scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.1)

    best_val_acc = 0.0
    best_state = None
    patience_counter = 0

    for epoch in range(config["num_epochs"]):
        model.train()
        train_loss = 0.0
        train_correct = 0
        train_total = 0
        for data, target in train_loader:
            data, target = data.to(device), target.to(device).long()
            optimizer.zero_grad()
            out = model(data)
            loss = criterion(out, target)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
            _, pred = out.max(1)
            train_total += target.size(0)
            train_correct += pred.eq(target).sum().item()

        model.eval()
        val_correct = 0
        val_total = 0
        with torch.no_grad():
            for data, target in val_loader:
                data, target = data.to(device), target.to(device).long()
                out = model(data)
                _, pred = out.max(1)
                val_total += target.size(0)
                val_correct += pred.eq(target).sum().item()

        val_acc = 100.0 * val_correct / val_total
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1

        if (epoch + 1) % 5 == 0 or epoch == 0:
            print(f"  Epoch {epoch+1}/{config['num_epochs']}  loss={train_loss/len(train_loader):.4f}  train_acc={100.*train_correct/train_total:.2f}%  val_acc={val_acc:.2f}%")

        if patience_counter >= config["patience"]:
            print(f"  Early stop at epoch {epoch+1}")
            break
        if scheduler is not None:
            scheduler.step()

    if best_state is not None:
        model.load_state_dict(best_state)
    return model, best_val_acc

## 7. Train all 3 ensemble models

In [11]:
print("Training binary ensemble (3 models)")
print("=" * 60)

trained_models = []
for i, model_class in enumerate(ENSEMBLE_CLASSES):
    name = f"ensemble_model_{i+1}"
    print(f"\n[{i+1}/3] {name}")
    model, val_acc = train_ensemble_model(
        model_class, name, train_loader, val_loader, device, CONFIG, class_weights=class_weights
    )
    path = OUTPUT_DIR / "models" / f"{name}.pth"
    torch.save(model.state_dict(), path)
    print(f"  Saved: {path}  (best val_acc={val_acc:.2f}%)")
    trained_models.append(model)

print("\n✅ All 3 models trained and saved")

Training binary ensemble (3 models)

[1/3] ensemble_model_1
Downloading: "https://download.pytorch.org/models/mobilenet_v3_large-5c1a4163.pth" to /Users/alokdwivedi/.cache/torch/hub/checkpoints/mobilenet_v3_large-5c1a4163.pth


100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 21.1M/21.1M [00:01<00:00, 16.5MB/s]


  Epoch 1/30  loss=0.5861  train_acc=68.90%  val_acc=70.50%
  Epoch 5/30  loss=0.2419  train_acc=88.84%  val_acc=80.28%
  Epoch 10/30  loss=0.1081  train_acc=95.41%  val_acc=80.44%
  Epoch 15/30  loss=0.0658  train_acc=97.38%  val_acc=83.28%
  Epoch 20/30  loss=0.0525  train_acc=97.86%  val_acc=82.65%
  Early stop at epoch 21
  Saved: /Users/alokdwivedi/dev/github/3amwave/oral-cavity-classifier/ml/training/notebooks/results/binary_ensemble/run_20260130_135331/models/ensemble_model_1.pth  (best val_acc=83.75%)

[2/3] ensemble_model_2
  Epoch 1/30  loss=0.5857  train_acc=69.24%  val_acc=74.76%
  Epoch 5/30  loss=0.2244  train_acc=89.76%  val_acc=79.97%
  Epoch 10/30  loss=0.0901  train_acc=96.19%  val_acc=80.91%
  Epoch 15/30  loss=0.0525  train_acc=97.86%  val_acc=82.49%
  Epoch 20/30  loss=0.0474  train_acc=97.66%  val_acc=82.81%
  Epoch 25/30  loss=0.0386  train_acc=98.33%  val_acc=83.12%
  Early stop at epoch 29
  Saved: /Users/alokdwivedi/dev/github/3amwave/oral-cavity-classifier/ml

## 8. Convert PyTorch → ONNX

In [20]:
def convert_binary_ensemble_to_onnx(pytorch_path, onnx_path, device, input_shape=(1, 3, 224, 224)):
    """Convert one binary ensemble .pth to ONNX (num_classes=2). Output is a single .onnx file (no .onnx.data)."""
    pytorch_path = Path(pytorch_path)
    onnx_path = Path(onnx_path)
    name = pytorch_path.stem

    if "ensemble_model_1" in name:
        model_class = EnsembleModel1
    elif "ensemble_model_2" in name:
        model_class = EnsembleModel2
    elif "ensemble_model_3" in name:
        model_class = EnsembleModel3
    else:
        model_class = ConservativeMobileNetV3

    model = model_class(num_classes=NUM_CLASSES)
    model.load_state_dict(torch.load(pytorch_path, map_location=device))
    model.eval()
    model.to(device)

    dummy = torch.randn(input_shape).to(device)
    # Use opset 18 to avoid version-converter downgrade errors (axes_input_to_attribute);
    # ONNX Runtime (flutter_onnxruntime) supports opset 18.
    torch.onnx.export(
        model,
        dummy,
        onnx_path,
        export_params=True,
        opset_version=18,
        do_constant_folding=True,
        input_names=["input"],
        output_names=["output"],
        dynamic_axes={"input": {0: "batch_size"}, "output": {0: "batch_size"}},
    )

    # Ensure single-file ONNX (no external .onnx.data): load and re-save with inlined data
    if HAS_ONNX:
        onnx_model = onnx.load(str(onnx_path))
        onnx.save_model(onnx_model, str(onnx_path), save_as_external_data=False)
        data_file = onnx_path.with_suffix(onnx_path.suffix + ".data")
        if data_file.exists():
            data_file.unlink()
    return True


onnx_dir = OUTPUT_DIR / "onnx"
for i in range(1, 4):
    name = f"ensemble_model_{i}"
    pth = OUTPUT_DIR / "models" / f"{name}.pth"
    onnx_path = onnx_dir / f"{name}.onnx"
    if pth.exists():
        convert_binary_ensemble_to_onnx(pth, onnx_path, device)
        print(f"✅ {name}.onnx (single file)")

print(f"\n📁 ONNX models in: {onnx_dir}")

W0201 18:32:21.998000 21091 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'input' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0, sampling_ratio: 'int' = -1, aligned: 'bool' = False). Treating as an Input.
W0201 18:32:21.999000 21091 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'boxes' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0, sampling_ratio: 'int' = -1, aligned: 'bool' = False). Treating as an Input.
W0201 18:32:21.999000 21091 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'input' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0). Treating as an Input.
W0201 18:32:22.000000 21091 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'boxes' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0). Treating as an Input.


[torch.onnx] Obtain model graph for `EnsembleModel1([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `EnsembleModel1([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decomposition...
[torch.onnx] Run decomposition... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
Applied 93 of general pattern rewrite rules.
✅ ensemble_model_1.onnx (single file)


W0201 18:32:25.781000 21091 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'input' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0, sampling_ratio: 'int' = -1, aligned: 'bool' = False). Treating as an Input.
W0201 18:32:25.782000 21091 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'boxes' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0, sampling_ratio: 'int' = -1, aligned: 'bool' = False). Treating as an Input.
W0201 18:32:25.783000 21091 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'input' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0). Treating as an Input.
W0201 18:32:25.783000 21091 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'boxes' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0). Treating as an Input.


[torch.onnx] Obtain model graph for `EnsembleModel2([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `EnsembleModel2([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decomposition...
[torch.onnx] Run decomposition... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
Applied 93 of general pattern rewrite rules.
✅ ensemble_model_2.onnx (single file)


W0201 18:32:29.456000 21091 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'input' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0, sampling_ratio: 'int' = -1, aligned: 'bool' = False). Treating as an Input.
W0201 18:32:29.456000 21091 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'boxes' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0, sampling_ratio: 'int' = -1, aligned: 'bool' = False). Treating as an Input.
W0201 18:32:29.457000 21091 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'input' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0). Treating as an Input.
W0201 18:32:29.457000 21091 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'boxes' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0). Treating as an Input.


[torch.onnx] Obtain model graph for `EnsembleModel3([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `EnsembleModel3([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decomposition...
[torch.onnx] Run decomposition... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
Applied 93 of general pattern rewrite rules.
✅ ensemble_model_3.onnx (single file)

📁 ONNX models in: /Users/alokdwivedi/dev/github/3amwave/oral-cavity-classifier/ml/training/notebooks/results/binary_ensemble/run_20260130_135331/onnx


## 9. Evaluate on frozen T5, TB, TM

In [13]:
from pathlib import Path

RUN_DIR = "run_20260130_135331"
BASE_PATH = Path("/Users/alokdwivedi/dev/github/3amwave/oral-cavity-classifier/ml/training/notebooks/results/binary_ensemble")

# The / operator with Path objects joins them intelligently
OUTPUT_DIR = BASE_PATH / RUN_DIR

print(OUTPUT_DIR)

/Users/alokdwivedi/dev/github/3amwave/oral-cavity-classifier/ml/training/notebooks/results/binary_ensemble/run_20260130_135331


In [14]:
# Run evaluate_on_frozen_sets.py for binary_onnx_ensemble
import subprocess

onnx_dir = OUTPUT_DIR / "onnx"
if (onnx_dir / "ensemble_model_1.onnx").exists():
    out_json = OUTPUT_DIR / "evaluation_frozen_T5_TB_TM.json"
    cmd = [
        "python", str(NOTEBOOK_DIR / "evaluate_on_frozen_sets.py"),
        "--model", str(onnx_dir),
        "--model_type", "binary_onnx_ensemble",
        "--output", str(out_json),
        "--frozen_dir", str(FROZEN_DIR),
    ]
    print("Running:", " ".join(cmd))
    subprocess.run(cmd, cwd=str(NOTEBOOK_DIR))
    if out_json.exists():
        with open(out_json) as f:
            res = json.load(f)
        print("\nFrozen set results:")
        for k in ["T5", "TB", "TM"]:
            if k in res:
                m = res[k]
                print(f"  {k}: Acc={m.get('accuracy',0)*100:.2f}%  Sens={m.get('sensitivity',0)*100:.2f}%  Spec={m.get('specificity',0)*100:.2f}%  n={m.get('n',0)}")
else:
    print("Skip: ONNX files not found. Run conversion cell first.")
    print("Manual: python evaluate_on_frozen_sets.py --model <path_to_onnx_dir> --model_type binary_onnx_ensemble --output results.json")

Running: python /Users/alokdwivedi/dev/github/3amwave/oral-cavity-classifier/ml/training/notebooks/evaluate_on_frozen_sets.py --model /Users/alokdwivedi/dev/github/3amwave/oral-cavity-classifier/ml/training/notebooks/results/binary_ensemble/run_20260130_135331/onnx --model_type binary_onnx_ensemble --output /Users/alokdwivedi/dev/github/3amwave/oral-cavity-classifier/ml/training/notebooks/results/binary_ensemble/run_20260130_135331/evaluation_frozen_T5_TB_TM.json --frozen_dir /Users/alokdwivedi/dev/github/3amwave/oral-cavity-classifier/ml/training/notebooks/frozen_test_sets
Running on T5...
Running on TB...
Running on TM...

Frozen test sets: T5, TB, TM
T5 (5-class test):  n=484  Acc=79.55%  Sens=79.59%  Spec=79.50%
TB (binary test):   n=246  Acc=93.09%  Sens=90.98%  Spec=95.58%
TM (mixed):         n=730  Acc=84.11%  Sens=83.60%  Spec=84.66%
TM per-source:
  Source A (5-class): Acc=79.55%  Sens=79.59%  Spec=79.50%
  Source B (binary): Acc=93.09%  Sens=90.98%  Spec=95.58%

Results saved

## 9b. Expanded evaluation metrics (binary ensemble)

Use the frozen sets (T5/TB/TM) to report the following metrics for **model comparison and threshold selection**:

- AUROC, AUPRC
- Sensitivity at fixed specificity (e.g., 80/85/90%)
- Specificity at fixed sensitivity (e.g., 90/95%)
- Confusion matrix + Sens/Spec/PPV/NPV at the chosen operating threshold
- Calibration: ECE + Brier score

These are the metrics used to select `T_low_referral`, `T_high_referral`, and `T_very_high` thresholds for the app.


In [17]:
# Metrics helpers for binary ensemble evaluation (requires per-sample y_true and y_prob).
from sklearn.metrics import roc_auc_score, average_precision_score, confusion_matrix, brier_score_loss
import numpy as np

def binary_metrics(y_true, y_prob, threshold=0.5, spec_targets=(0.8, 0.85, 0.9), sens_targets=(0.9, 0.95)):
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob).astype(float)
    y_pred = (y_prob >= threshold).astype(int)

    auroc = roc_auc_score(y_true, y_prob)
    auprc = average_precision_score(y_true, y_prob)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    sensitivity = tp / max(tp + fn, 1)
    specificity = tn / max(tn + fp, 1)
    ppv = tp / max(tp + fp, 1)
    npv = tn / max(tn + fn, 1)
    brier = brier_score_loss(y_true, y_prob)

    # Sens@Spec and Spec@Sens
    thresholds = np.linspace(0.0, 1.0, 1001)
    sens_at_spec = {}
    spec_at_sens = {}
    for t in thresholds:
        yp = (y_prob >= t).astype(int)
        tn, fp, fn, tp = confusion_matrix(y_true, yp).ravel()
        sens = tp / max(tp + fn, 1)
        spec = tn / max(tn + fp, 1)
        for target in spec_targets:
            if spec >= target and target not in sens_at_spec:
                sens_at_spec[target] = (sens, t)
        for target in sens_targets:
            if sens >= target and target not in spec_at_sens:
                spec_at_sens[target] = (spec, t)

    return {
        "auroc": auroc,
        "auprc": auprc,
        "threshold": threshold,
        "confusion": {"tp": int(tp), "fp": int(fp), "fn": int(fn), "tn": int(tn)},
        "sensitivity": sensitivity,
        "specificity": specificity,
        "ppv": ppv,
        "npv": npv,
        "brier": brier,
        "sens_at_spec": sens_at_spec,
        "spec_at_sens": spec_at_sens,
    }

# Example usage (replace with arrays from frozen-set inference):
# y_true, y_prob = ...
# metrics = binary_metrics(y_true, y_prob, threshold=0.5)
# metrics


In [18]:
# Cell 9B-RUN: Actually run the expanded metrics on frozen test sets
import onnxruntime as ort
from pathlib import Path
import numpy as np
from PIL import Image

# Setup
FROZEN_DIR = NOTEBOOK_DIR / "frozen_test_sets"
BINARY_ONNX_DIR = OUTPUT_DIR / "onnx"

# Load frozen test set paths and labels
from frozen_test_sets import get_tm_paths_labels_and_sources
tm_paths, tm_labels, tm_sources = get_tm_paths_labels_and_sources(FROZEN_DIR)

# Preprocessing function
def preprocess(path):
    img = Image.open(path).convert("RGB").resize((224, 224))
    arr = np.array(img).astype(np.float32) / 255.0
    arr = (arr - [0.485, 0.456, 0.406]) / [0.229, 0.224, 0.225]
    return np.transpose(arr, (2, 0, 1))[np.newaxis, ...].astype(np.float32)

# Load binary ensemble
sessions = [
    ort.InferenceSession(str(BINARY_ONNX_DIR / f"ensemble_model_{i}.onnx"))
    for i in [1, 2, 3]
]
input_name = sessions[0].get_inputs()[0].name
weights = [0.30, 0.40, 0.30]

# Run inference and collect probabilities
y_true = []
y_prob = []

print(f"Running inference on TM ({len(tm_paths)} samples)...")
for i, (path, label) in enumerate(zip(tm_paths, tm_labels)):
    x = preprocess(path)
    logits = sum(sess.run(None, {input_name: x})[0] * w for sess, w in zip(sessions, weights))
    exp_logits = np.exp(logits - np.max(logits))
    prob_cancer = float(exp_logits[0, 1] / exp_logits.sum())
    
    y_true.append(label)
    y_prob.append(prob_cancer)
    
    if (i + 1) % 100 == 0:
        print(f"  {i + 1}/{len(tm_paths)} done")

print("Done!\n")

# Now run the expanded metrics
metrics = binary_metrics(y_true, y_prob, threshold=0.5)

print("=" * 60)
print("EXPANDED METRICS FOR THRESHOLD SELECTION")
print("=" * 60)
print(f"AUROC: {metrics['auroc']:.3f}  (1.0 = perfect, 0.5 = random)")
print(f"AUPRC: {metrics['auprc']:.3f}")
print(f"Brier Score: {metrics['brier']:.4f}  (lower = better calibrated)")
print()

print("At default threshold (0.5):")
print(f"  Sensitivity: {metrics['sensitivity']:.1%}")
print(f"  Specificity: {metrics['specificity']:.1%}")
print()

print("SENSITIVITY at fixed SPECIFICITY:")
print("(Use these to pick thresholds that limit false alarms)")
for spec_target, (sens, thresh) in sorted(metrics['sens_at_spec'].items()):
    print(f"  If Specificity ≥ {spec_target:.0%}: Sensitivity = {sens:.1%} @ threshold = {thresh:.3f}")
print()

print("SPECIFICITY at fixed SENSITIVITY:")
print("(Use these to pick thresholds that catch most cancers)")
for sens_target, (spec, thresh) in sorted(metrics['spec_at_sens'].items(), reverse=True):
    print(f"  If Sensitivity ≥ {sens_target:.0%}: Specificity = {spec:.1%} @ threshold = {thresh:.3f}")
print()

print("=" * 60)
print("RECOMMENDED THRESHOLDS FOR APP:")
print("=" * 60)
if 0.95 in metrics['spec_at_sens']:
    spec, thresh = metrics['spec_at_sens'][0.95]
    print(f"T_low_referral  = {thresh:.2f}  (catches 95% of cancers, spec={spec:.1%})")
if 0.90 in metrics['spec_at_sens']:
    spec, thresh = metrics['spec_at_sens'][0.90]
    print(f"T_high_referral = {thresh:.2f}  (catches 90% of cancers, spec={spec:.1%})")
print(f"T_very_high     = 0.70  (high confidence predictions)")

Running inference on TM (730 samples)...
  100/730 done
  200/730 done
  300/730 done
  400/730 done
  500/730 done
  600/730 done
  700/730 done
Done!

EXPANDED METRICS FOR THRESHOLD SELECTION
AUROC: 0.920  (1.0 = perfect, 0.5 = random)
AUPRC: 0.933
Brier Score: 0.1369  (lower = better calibrated)

At default threshold (0.5):
  Sensitivity: 83.6%
  Specificity: 84.7%

SENSITIVITY at fixed SPECIFICITY:
(Use these to pick thresholds that limit false alarms)
  If Specificity ≥ 80%: Sensitivity = 86.2% @ threshold = 0.311
  If Specificity ≥ 85%: Sensitivity = 83.3% @ threshold = 0.535
  If Specificity ≥ 90%: Sensitivity = 77.2% @ threshold = 0.880

SPECIFICITY at fixed SENSITIVITY:
(Use these to pick thresholds that catch most cancers)
  If Sensitivity ≥ 95%: Specificity = 0.0% @ threshold = 0.000
  If Sensitivity ≥ 90%: Specificity = 0.0% @ threshold = 0.000

RECOMMENDED THRESHOLDS FOR APP:
T_low_referral  = 0.00  (catches 95% of cancers, spec=0.0%)
T_high_referral = 0.00  (catches 90% o

## 10. Summary and next steps

In [ ]:
print("Binary ensemble run complete.")
print(f"  PyTorch: {OUTPUT_DIR / 'models'}")
print(f"  ONNX:    {OUTPUT_DIR / 'onnx'}")
print("\nCopy single-file ONNX to app assets:")
print(f"  cp {OUTPUT_DIR / 'onnx'}/ensemble_model_1.onnx <repo>/assets/binary_ensemble/")
print(f"  cp {OUTPUT_DIR / 'onnx'}/ensemble_model_2.onnx <repo>/assets/binary_ensemble/")
print(f"  cp {OUTPUT_DIR / 'onnx'}/ensemble_model_3.onnx <repo>/assets/binary_ensemble/")
print("(No .onnx.data files needed; each .onnx is self-contained.)")

Binary ensemble run complete.
  PyTorch: /Users/alokdwivedi/dev/github/3amwave/oral-cavity-classifier/ml/training/notebooks/results/binary_ensemble/run_20260130_135331/models
  ONNX:    /Users/alokdwivedi/dev/github/3amwave/oral-cavity-classifier/ml/training/notebooks/results/binary_ensemble/run_20260130_135331/onnx

Copy single-file ONNX to app assets:
  cp /Users/alokdwivedi/dev/github/3amwave/oral-cavity-classifier/ml/training/notebooks/results/binary_ensemble/run_20260130_135331/onnx/ensemble_model_1.onnx <repo>/assets/binary_ensemble/
  cp /Users/alokdwivedi/dev/github/3amwave/oral-cavity-classifier/ml/training/notebooks/results/binary_ensemble/run_20260130_135331/onnx/ensemble_model_2.onnx <repo>/assets/binary_ensemble/
  cp /Users/alokdwivedi/dev/github/3amwave/oral-cavity-classifier/ml/training/notebooks/results/binary_ensemble/run_20260130_135331/onnx/ensemble_model_3.onnx <repo>/assets/binary_ensemble/
(No .onnx.data files needed; each .onnx is self-contained.)


## 11. Test on ml/test_data (binary vs 5-class agreement)

Run both the **binary ensemble** (this notebook’s ONNX) and the **5-class ensemble** on `ml/test_data` and report where they agree or disagree on the referral decision (non-cancer vs cancer). Useful to sanity-check dual-ensemble behaviour before deploying.

In [25]:
# Test on ml/test_data: binary ensemble vs 5-class ensemble agreement report
import sys
from pathlib import Path

TEST_DATA_DIR = NOTEBOOK_DIR.parent.parent / "test_data"  # ml/test_data
BINARY_ONNX_DIR = OUTPUT_DIR / "onnx"  # this notebook's binary ensemble
REPO_ROOT = NOTEBOOK_DIR.parent.parent.parent
FIVECLASS_ONNX_DIR = REPO_ROOT / "assets"  # 5-class ensemble_model_1,2,3.onnx (change if elsewhere)

def collect_test_images(folder, exts=("*.jpg", "*.jpeg", "*.png", "*.JPG", "*.JPEG", "*.PNG")):
    folder = Path(folder)
    out = []
    if folder.exists():
        for ext in exts:
            out.extend(sorted(folder.glob(ext)))
    return [str(p) for p in out]

def preprocess_onnx(path, image_size=(224, 224)):
    from PIL import Image
    img = Image.open(path).convert("RGB")
    img = img.resize(image_size)
    arr = np.array(img).astype(np.float32) / 255.0
    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])
    arr = (arr - mean) / std
    arr = np.transpose(arr, (2, 0, 1))
    return np.expand_dims(arr, axis=0).astype(np.float32)

# 5-class index -> label (same as app)
FIVECLASS_NAMES = ["Benign", "Healthy", "OCA", "OPMD", "Unknown"]
CANCER_CLASS_INDICES = {2, 3}  # OCA, OPMD -> binary 1
ENSEMBLE_WEIGHTS = [0.30, 0.40, 0.30]
threshold = float(CONFIG.get("binary_threshold", 0.5))

def expected_from_filename(path):
    name = Path(path).stem.lower()
    if "benign" in name or "healthy" in name:
        return 0, Path(path).stem
    if "oca" in name or "opmd" in name:
        return 1, Path(path).stem
    return None, Path(path).stem  # unknown

if not TEST_DATA_DIR.exists():
    print(f"⚠️ Test data dir not found: {TEST_DATA_DIR}")
else:
    paths = collect_test_images(TEST_DATA_DIR)
    if not paths:
        print(f"⚠️ No images in {TEST_DATA_DIR}")
    else:
        try:
            import onnxruntime as ort
        except ImportError:
            print("⚠️ onnxruntime required. pip install onnxruntime")
        else:
            # Load binary ensemble (this run)
            binary_sessions = []
            for name in ["ensemble_model_1.onnx", "ensemble_model_2.onnx", "ensemble_model_3.onnx"]:
                p = BINARY_ONNX_DIR / name
                if not p.exists():
                    print(f"⚠️ Binary ONNX not found: {p}. Run section 8 first.")
                    break
                binary_sessions.append(ort.InferenceSession(str(p)))
            else:
                # Load 5-class ensemble (assets or custom path)
                five_sessions = []
                for name in ["ensemble_model_1.onnx", "ensemble_model_2.onnx", "ensemble_model_3.onnx"]:
                    p = FIVECLASS_ONNX_DIR / name
                    if not p.exists():
                        print(f"⚠️ 5-class ONNX not found: {p}. Set FIVECLASS_ONNX_DIR to your 5-class ONNX folder.")
                        break
                    five_sessions.append(ort.InferenceSession(str(p)))
                else:
                    input_name_bin = binary_sessions[0].get_inputs()[0].name
                    input_name_5 = five_sessions[0].get_inputs()[0].name

                    rows = []
                    for path in paths:
                        x = preprocess_onnx(path)
                        # Binary
                        logits_bin = np.zeros((1, 2))
                        for sess, w in zip(binary_sessions, ENSEMBLE_WEIGHTS):
                            out = sess.run(None, {input_name_bin: x})[0]
                            logits_bin += out * w
                        exp_bin = np.exp(logits_bin - np.max(logits_bin))
                        prob_cancer = float((exp_bin[0, 1] / exp_bin.sum()).item())
                        pred_bin = 1 if prob_cancer >= threshold else 0
                        # 5-class
                        logits_5 = np.zeros((1, 5))
                        for sess, w in zip(five_sessions, ENSEMBLE_WEIGHTS):
                            out = sess.run(None, {input_name_5: x})[0]
                            logits_5 += out * w
                        idx_5 = int(np.argmax(logits_5[0]))
                        label_5 = FIVECLASS_NAMES[idx_5]
                        pred_5_bin = 1 if idx_5 in CANCER_CLASS_INDICES else 0
                        agreed = pred_bin == pred_5_bin
                        exp_val, _ = expected_from_filename(path)
                        rows.append({
                            "file": Path(path).name,
                            "expected": exp_val,
                            "binary_pred": pred_bin,
                            "binary_prob": prob_cancer,
                            "5class_label": label_5,
                            "5class_binary": pred_5_bin,
                            "agreed": agreed,
                        })

                    # Report
                    agreed_count = sum(r["agreed"] for r in rows)
                    disagreed_count = len(rows) - agreed_count
                    print("=" * 80)
                    print("ml/test_data: Binary ensemble vs 5-class ensemble agreement")
                    print("=" * 80)
                    print(f"Binary ONNX: {BINARY_ONNX_DIR}")
                    print(f"5-class ONNX: {FIVECLASS_ONNX_DIR}")
                    print(f"Images: {len(rows)}")
                    print()
                    print(f"{'File':<35} {'Exp':<4} {'Bin':<4} {'P(cancer)':<10} {'5-class':<10} {'5→bin':<5} {'Agreed':<6}")
                    print("-" * 80)
                    for r in rows:
                        exp_s = str(r["expected"]) if r["expected"] is not None else "?"
                        print(f"{r['file']:<35} {exp_s:<4} {r['binary_pred']:<4} {r['binary_prob']:.3f}      {r['5class_label']:<10} {r['5class_binary']:<5} {'Yes' if r['agreed'] else 'No':<6}")
                    print("-" * 80)
                    print(f"Summary: {agreed_count} agreed, {disagreed_count} disagreed")
                    if disagreed_count > 0:
                        print("Disagreements:")
                        for r in rows:
                            if not r["agreed"]:
                                print(f"  {r['file']}: binary={r['binary_pred']} (P={r['binary_prob']:.3f}), 5-class={r['5class_label']} (→{r['5class_binary']})")
                    print("=" * 80)

ml/test_data: Binary ensemble vs 5-class ensemble agreement
Binary ONNX: /Users/alokdwivedi/dev/github/3amwave/oral-cavity-classifier/ml/training/notebooks/results/binary_ensemble/run_20260130_135331/onnx
5-class ONNX: /Users/alokdwivedi/dev/github/3amwave/oral-cavity-classifier/assets
Images: 10

File                                Exp  Bin  P(cancer)  5-class    5→bin Agreed
--------------------------------------------------------------------------------
benign_27.jpg                       0    0    0.017      Benign     0     Yes   
benign_42.jpg                       0    1    0.996      Benign     0     No    
healthy_C-4-6-2.jpg                 0    0    0.000      Healthy    0     Yes   
healthy_C-5-6-1.jpg                 0    0    0.006      Healthy    0     Yes   
oca_27.jpg                          1    1    0.989      OCA        1     Yes   
oca_34.jpg                          1    1    1.000      Benign     0     No    
opmd_K-005-03.jpg                   1    0    0.235  

## 12. Binary vs 5-class agreement on frozen test sets (T5, TB, TM)

Run both ensembles on the **frozen test sets** (T5: 5-class test, TB: binary-only test, TM: mixed). Report **running progress** of disagreements as each set is processed, then a **final summary** at the end. Uses `frozen_test_sets` saved to `FROZEN_DIR`.

In [ ]:
# Binary vs 5-class agreement on frozen test sets (T5, TB, TM) with running progress
import sys
from pathlib import Path

sys.path.insert(0, str(NOTEBOOK_DIR))
from frozen_test_sets import (
    get_t5_binary_paths_labels,
    get_tb_paths_labels,
    get_tm_paths_labels_and_sources,
)

REPO_ROOT = NOTEBOOK_DIR.parent.parent.parent
BINARY_ONNX_DIR = OUTPUT_DIR / "onnx"
FIVECLASS_ONNX_DIR = REPO_ROOT / "assets"
BATCH_SIZE = 32
PROGRESS_EVERY = 50  # print progress every N samples
IMAGE_SIZE = (224, 224)
FIVECLASS_NAMES = ["Benign", "Healthy", "OCA", "OPMD", "Unknown"]
CANCER_CLASS_INDICES = {2, 3}
ENSEMBLE_WEIGHTS = [0.30, 0.40, 0.30]
threshold = float(CONFIG.get("binary_threshold", 0.5))

def preprocess_onnx(path, image_size=(224, 224)):
    from PIL import Image
    img = Image.open(path).convert("RGB")
    img = img.resize(image_size)
    arr = np.array(img).astype(np.float32) / 255.0
    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])
    arr = (arr - mean) / std
    arr = np.transpose(arr, (2, 0, 1))
    return np.expand_dims(arr, axis=0).astype(np.float32)

try:
    import onnxruntime as ort
except ImportError:
    print("⚠️ onnxruntime required. pip install onnxruntime")
else:
    # Load binary ensemble
    binary_sessions = []
    for name in ["ensemble_model_1.onnx", "ensemble_model_2.onnx", "ensemble_model_3.onnx"]:
        p = BINARY_ONNX_DIR / name
        if not p.exists():
            print(f"⚠️ Binary ONNX not found: {p}. Run section 8 first.")
            break
        binary_sessions.append(ort.InferenceSession(str(p)))
    else:
        # Load 5-class ensemble
        five_sessions = []
        for name in ["ensemble_model_1.onnx", "ensemble_model_2.onnx", "ensemble_model_3.onnx"]:
            p = FIVECLASS_ONNX_DIR / name
            if not p.exists():
                print(f"⚠️ 5-class ONNX not found: {p}. Set FIVECLASS_ONNX_DIR.")
                break
            five_sessions.append(ort.InferenceSession(str(p)))
        else:
            input_name_bin = binary_sessions[0].get_inputs()[0].name
            input_name_5 = five_sessions[0].get_inputs()[0].name

            def run_one_set(name, paths, labels, sources=None):
                n = len(paths)
                agreed, disagreed = 0, 0
                last_printed = 0
                for i in range(0, n, BATCH_SIZE):
                    batch_paths = paths[i : i + BATCH_SIZE]
                    batch_labels = labels[i : i + BATCH_SIZE]
                    batch_arrays = [preprocess_onnx(p, IMAGE_SIZE) for p in batch_paths]
                    batch_input = np.concatenate(batch_arrays, axis=0).astype(np.float32)
                    # Binary
                    logits_bin = np.zeros((len(batch_paths), 2))
                    for sess, w in zip(binary_sessions, ENSEMBLE_WEIGHTS):
                        out = sess.run(None, {input_name_bin: batch_input})[0]
                        logits_bin += out * w
                    exp_bin = np.exp(logits_bin - np.max(logits_bin, axis=1, keepdims=True))
                    prob_cancer = (exp_bin[:, 1] / exp_bin.sum(axis=1)).flatten()
                    pred_bin = (prob_cancer >= 0.5).astype(int)
                    # 5-class
                    logits_5 = np.zeros((len(batch_paths), 5))
                    for sess, w in zip(five_sessions, ENSEMBLE_WEIGHTS):
                        out = sess.run(None, {input_name_5: batch_input})[0]
                        logits_5 += out * w
                    idx_5 = np.argmax(logits_5, axis=1)
                    pred_5_bin = np.array([1 if c in CANCER_CLASS_INDICES else 0 for c in idx_5])
                    agree_batch = (pred_bin == pred_5_bin)
                    agreed += int(agree_batch.sum())
                    disagreed += int((~agree_batch).sum())
                    done = min(i + BATCH_SIZE, n)
                    if done >= last_printed + PROGRESS_EVERY or done == n:
                        print(f"  {name}: {done}/{n} processed, disagreements so far: {disagreed}")
                        last_printed = done
                return agreed, disagreed

            print("Frozen test sets: binary vs 5-class agreement (with running progress)")
            print(f"FROZEN_DIR: {FROZEN_DIR}")
            print(f"Binary ONNX: {BINARY_ONNX_DIR}")
            print(f"5-class ONNX: {FIVECLASS_ONNX_DIR}")
            print()

            results = {}
            # T5
            t5_paths, t5_labels = get_t5_binary_paths_labels(FROZEN_DIR)
            print(f"T5 (5-class test, binary-eligible): {len(t5_paths)} samples")
            a, d = run_one_set("T5", t5_paths, t5_labels)
            results["T5"] = {"total": len(t5_paths), "agreed": a, "disagreed": d}
            print(f"  T5 complete: {len(t5_paths)} total, {a} agreed, {d} disagreed\n")

            # TB
            tb_paths, tb_labels = get_tb_paths_labels(FROZEN_DIR)
            print(f"TB (binary-only test): {len(tb_paths)} samples")
            a, d = run_one_set("TB", tb_paths, tb_labels)
            results["TB"] = {"total": len(tb_paths), "agreed": a, "disagreed": d}
            print(f"  TB complete: {len(tb_paths)} total, {a} agreed, {d} disagreed\n")

            # TM
            tm_paths, tm_labels, _ = get_tm_paths_labels_and_sources(FROZEN_DIR)
            print(f"TM (mixed): {len(tm_paths)} samples")
            a, d = run_one_set("TM", tm_paths, tm_labels)
            results["TM"] = {"total": len(tm_paths), "agreed": a, "disagreed": d}
            print(f"  TM complete: {len(tm_paths)} total, {a} agreed, {d} disagreed\n")

            # Final summary
            print("=" * 60)
            print("FINAL SUMMARY – Binary vs 5-class agreement on frozen sets")
            print("=" * 60)
            print(f"{'Set':<6} {'Total':<8} {'Agreed':<8} {'Disagreed':<10} {'Agreement %':<12}")
            print("-" * 60)
            tot_total, tot_agreed, tot_disagreed = 0, 0, 0
            for name in ["T5", "TB", "TM"]:
                r = results[name]
                t, a, d = r["total"], r["agreed"], r["disagreed"]
                pct = (100.0 * a / t) if t else 0
                print(f"{name:<6} {t:<8} {a:<8} {d:<10} {pct:.1f}%")
                tot_total += t
                tot_agreed += a
                tot_disagreed += d
            print("-" * 60)
            overall_pct = (100.0 * tot_agreed / tot_total) if tot_total else 0
            print(f"{'Total':<6} {tot_total:<8} {tot_agreed:<8} {tot_disagreed:<10} {overall_pct:.1f}%")
            print("=" * 60)

Frozen test sets: binary vs 5-class agreement (with running progress)
FROZEN_DIR: /Users/alokdwivedi/dev/github/3amwave/oral-cavity-classifier/ml/training/notebooks/frozen_test_sets
Binary ONNX: /Users/alokdwivedi/dev/github/3amwave/oral-cavity-classifier/ml/training/notebooks/results/binary_ensemble/run_20260130_135331/onnx
5-class ONNX: /Users/alokdwivedi/dev/github/3amwave/oral-cavity-classifier/assets

T5 (5-class test, binary-eligible): 484 samples
  T5: 64/484 processed, disagreements so far: 14
  T5: 128/484 processed, disagreements so far: 26
  T5: 192/484 processed, disagreements so far: 30
  T5: 256/484 processed, disagreements so far: 40
  T5: 320/484 processed, disagreements so far: 53
  T5: 384/484 processed, disagreements so far: 60
  T5: 448/484 processed, disagreements so far: 70
  T5: 484/484 processed, disagreements so far: 77
  T5 complete: 484 total, 407 agreed, 77 disagreed

TB (binary-only test): 246 samples
  TB: 64/246 processed, disagreements so far: 12
  TB: 1